<a href="https://colab.research.google.com/github/mabrarbel-eng/DSC110/blob/main/week5_pandas_commented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import pandas, the Python library used to work with tabular data.
# Read the NHANES CSV dataset from the course GitHub repository into a DataFrame named df.
# A DataFrame is similar to a table in a database: rows are observations and columns are variables.
import pandas as pd

# DataFrame in Python is the equivalent of a table in SQL.
df = pd.read_csv("https://raw.githubusercontent.com/osoliman/DSC110/refs/heads/main/Datasets/nhanes_final.csv")

In [ ]:
# Display the first five rows of df by default.
# This is a quick way to preview the records and check the column names and values.
df.head()

In [ ]:
# Display the last five rows of df by default.
# This can help preview records at the end of the dataset.
df.tail()

In [ ]:
# Display the first 20 rows so we can inspect more observations at once.
df.head(20)

In [ ]:
# Display the last 10 rows so we can inspect a larger sample from the end of the dataset.
df.tail(10)

In [ ]:
# Return the dimensions of the DataFrame as a tuple: (number of rows, number of columns).
df.shape

In [ ]:
# Select the first value from df.shape to get the total number of rows/observations.
df.shape[0]

In [ ]:
# Select the second value from df.shape to get the total number of columns/variables.
df.shape[1]

In [ ]:
# Save the row count and column count in separate variables for easier reuse.
n_rows = df.shape[0]
n_cols = df.shape[1]

# Use an f-string to insert those values into a readable summary sentence.
print(f"My dataset contains {n_rows} rows and {n_cols} columns")

## Boolean

# A Boolean is a logical value: True or False. Comparisons produce Boolean results, and Booleans can also be used in arithmetic.

In [ ]:
# The == operator compares two values and returns a Boolean value (True or False).
# Since 1 + 1 is not equal to 3, this expression returns False.
1 + 1 == 3

In [ ]:
# This comparison is True because 1 + 1 equals 2.
1 + 1 == 2

In [ ]:
# In Python, Boolean True behaves like 1 in arithmetic, so True + True evaluates to 2.
True + True

In [ ]:
# In arithmetic, True behaves like 1 and False behaves like 0, so the result is 1.
True + False

## Data Cleaning

# This section identifies missing data, measures how much is missing, and demonstrates ways to handle it.

In [ ]:
# Preview the first five values in the bmi column to inspect body mass index data.
# Missing numeric values are commonly represented as NaN ("Not a Number").
df['bmi'].head()

In [ ]:
# isna() returns True for missing values and False for values that are present.
# head() displays the missing-value indicators for the first five BMI records.
df['bmi'].isna().head()

In [ ]:
# Sum the True values from isna() to count how many BMI entries are missing.
df['bmi'].isna().sum()

We'll create a new dataset for people over 18 years old and then count how many BMI values are missing in that subset.

# Filtering to adults lets us compare missing BMI data in a specific group with the full dataset.

In [ ]:
# Filter the dataset to keep only adults older than 18 years.
# The resulting subset is stored in df_adults; the original df remains available.
df_adults = df[df['age'] > 18]

In [ ]:
# Count how many rows are in the adult-only subset.
df_adults.shape[0]

In [ ]:
# Find the minimum age in df_adults to check the age filter.
# Because the condition is age > 18, the minimum should be greater than 18 when data are present.
df_adults['age'].min()

In [ ]:
# Count missing BMI values among adults only.
# Comparing this with the full-dataset count shows how missingness differs in the subset.
df_adults['bmi'].isna().sum()

Now it's your turn to do the same for education and have the output in an f-string.

# The next cells count missing education values in the adult subset and the full dataset, then print both counts.

In [ ]:
# Count the missing values in the education column for the adult-only dataset.
missing_ed_adults = df_adults['education'].isna().sum()

In [ ]:
# Count the missing values in the education column for the entire dataset.
missing_ed_all = df['education'].isna().sum()

In [ ]:
# Print both education missing-value counts using an f-string.
# The newline (\n) moves the adult-dataset count to the next line.
print(f"The number of rows with missing education in the entire dataset is {missing_ed_all},\n while it's only {missing_ed_adults} in the Adults dataset ")

In [ ]:
# Count missing values in every column across the entire DataFrame.
# The result is a Series showing one missing-value count for each variable.
df.isna().sum()

In [ ]:
# Calculate the proportion of missing values in each column.
# mean() treats True as 1 and False as 0, so multiplying by 100 converts proportions to percentages.
df.isna().mean()*100

In [ ]:
# Round each column's missing-value percentage to one decimal place for easier reading.
round(df.isna().mean()*100,1)

## Treating Missingness

Two approaches shown in this lesson:

1. Drop rows with missing values.
2. Fill missing values with the mean.

# The appropriate approach depends on the analysis and how much information would be lost or changed.

In [ ]:
# Drop every row that contains at least one missing value in any column.
# Store the complete-case dataset in df_complete, leaving df unchanged at this step.
df_complete = df.dropna()

In [ ]:
# Check how many rows remain after dropping rows with any missing value.
df_complete.shape[0]

In [ ]:
# Recalculate the missing-value percentage in each column of df_complete.
# Because rows with any missing value were dropped, each column should have 0% missingness.
df_complete.isna().mean()*100

In [ ]:
# Drop only rows where BMI is missing, rather than requiring every column to be complete.
# This keeps rows with missing values in other columns if their BMI is present.
df_complete_bmi = df.dropna(subset = ['bmi'])

In [ ]:
# Verify that no missing BMI values remain in df_complete_bmi.
# The expected result is 0 because rows with missing BMI were removed.
df_complete_bmi['bmi'].isna().sum()

In [ ]:
# Calculate the mean systolic blood pressure, ignoring missing values by default.
# Save the mean so it can be used to fill missing systolic blood pressure entries.
mean_bp = df['systolic_bp'].mean()

In [ ]:
# Display the mean systolic blood pressure rounded to two decimal places.
# round() affects the displayed value here; it does not change mean_bp itself.
print(round(mean_bp,2))

In [ ]:
# Replace missing values in systolic_bp with the mean calculated above.
# Assign the filled column back to df so the DataFrame now contains the imputed values.
df['systolic_bp'] = df['systolic_bp'].fillna(mean_bp)

In [ ]:
# Count missing values in systolic_bp after filling them with the mean.
# A result of 0 indicates that no missing values remain in this column.
df['systolic_bp'].isna().sum()

## Transform Variables

# This section creates categorical BMI labels from numeric BMI values using both a custom function and pandas cut().

In [ ]:
# Define a function that converts one BMI number into a BMI category.
# The if/elif statements check the BMI ranges in order and return the matching label.
def bmi_check(bmi):
  if bmi < 18.5:
    bmi_result = "Underweight"
  elif bmi < 25:
    bmi_result = "Normal"
  elif bmi < 30:
    bmi_result = "Overweight"
  else:
    bmi_result = "Obese"
  return bmi_result

In [ ]:
# Apply bmi_check() to every value in the bmi column.
# Store the returned category for each row in a new column named bmi_category.
df['bmi_category'] = df['bmi'].apply(bmi_check)

In [ ]:
# Count how many records fall into each category in bmi_category.
# value_counts() excludes missing category values by default.
df['bmi_category'].value_counts()

In [ ]:
# Use pandas cut() to group BMI values into labeled intervals.
# bins defines the interval boundaries, and labels names each interval.
# Values outside the bins, including missing BMI values, receive a missing category.
df['bmi_category2'] = pd.cut(df['bmi'],
                             bins = [0, 18.5, 25, 30, 100],
                             labels = ["Underweight", "Normal", "Overweight", "Obese"])

In [ ]:
# Count the number of observations in each category created by pd.cut().
# Missing values are excluded from these counts by default.
df['bmi_category2'].value_counts()

In [ ]:
# Count each BMI category and include missing category values in the output.
# dropna=False makes it possible to see how many records did not receive a category.
df['bmi_category2'].value_counts(dropna = False)

Next time, we'll start from transforming/decoding categorical data.

# Categorical decoding turns coded values into readable labels that are easier to interpret.